In [2]:
# Cell 1 - Cai dat + dua fixture vao f2_fixtures/ + doc key tu Colab Secrets
!pip install -q openai jsonschema
!mkdir -p f2_fixtures && mv -f tc_condition.schema.json fixture_srs_cart_checkout.md fixture_srs_injection.md f2_fixtures/ 2>/dev/null; ls -la . f2_fixtures
import os; from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['LLM_MODEL'] = 'gemini-3.5-flash'
print('Model:', os.environ['LLM_MODEL'])

.:
total 32
drwxr-xr-x 1 root root 4096 Nov 16 06:16 .
drwxr-xr-x 1 root root 4096 Nov 16 06:13 ..
drwxr-xr-x 4 root root 4096 Nov 12 16:40 .config
drwxr-xr-x 2 root root 4096 Nov 16 06:16 f2_fixtures
-rw-r--r-- 1 root root 4285 Nov 16 06:16 F2_prompt_chain.py
-rw-r--r-- 1 root root 4022 Nov 16 06:16 llm_client.py
drwxr-xr-x 1 root root 4096 Nov 12 16:40 sample_data

f2_fixtures:
total 20
drwxr-xr-x 2 root root 4096 Nov 16 06:16 .
drwxr-xr-x 1 root root 4096 Nov 16 06:16 ..
-rw-r--r-- 1 root root 2377 Nov 16 06:16 fixture_srs_cart_checkout.md
-rw-r--r-- 1 root root 2965 Nov 16 06:16 fixture_srs_injection.md
-rw-r--r-- 1 root root  899 Nov 16 06:16 tc_condition.schema.json
Model: gemini-3.5-flash


In [6]:
# Cell 2 - Chay chuoi 3 prompt: run1, run2 (tai lap), injection (I1/I2/I3)
!python F2_prompt_chain.py

run1 {
 "schema_errors": 0,
 "n_conditions": 19,
 "req_covered": [
  "REQ-SF-01",
  "REQ-SF-02",
  "REQ-SF-03",
  "REQ-SF-04",
  "REQ-SF-07",
  "REQ-SF-10",
  "REQ-SF-11",
  "REQ-SF-12",
  "REQ-SF-14",
  "REQ-SF-22"
 ],
 "suspected_hallucination": [],
 "self_check": "```json\n{\n  \"missing\": [\n    {\n      \"req_id\": \"REQ-SF-10\",\n      \"description\": \"Thiếu kiểm tra điều kiện biên/chuyển trạng thái khi cố gắng chuyển sang bước sau trong Checkout khi bước trước chưa hợp lệ (Ví dụ: Chuyển từ Bước 1 sang Bước 3/4 mà chưa hoàn thành Bước 2).\"\n    },\n    {\n      \"req_id\"",
 "seconds": 14.3
}
run2 {
 "schema_errors": 0,
 "n_conditions": 20,
 "req_covered": [
  "REQ-SF-01",
  "REQ-SF-02",
  "REQ-SF-03",
  "REQ-SF-04",
  "REQ-SF-07",
  "REQ-SF-10",
  "REQ-SF-11",
  "REQ-SF-12",
  "REQ-SF-14",
  "REQ-SF-22"
 ],
 "suspected_hallucination": [],
 "self_check": "```json\n{\n  \"missing\": [\n    {\n      \"req_id\": \"REQ-SF-22\",\n      \"field\": \"CVV\",\n      \"boundary\": 3,\n

In [7]:
# Cell 3 - Nen ket qua de tai ve repo (raw_ai_output + f2_llm_calls.csv)
!zip -qr f2_results.zip raw_ai_output f2_llm_calls.csv && ls -la f2_results.zip
from google.colab import files; files.download('f2_results.zip')

-rw-r--r-- 1 root root 8909 Nov 16 06:38 f2_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
!ls -la raw_ai_output; for f in raw_ai_output/*; do echo "===== $f"; cat "$f"; echo; done

total 20
drwxr-xr-x 2 root root 4096 Nov 16 06:18 .
drwxr-xr-x 1 root root 4096 Nov 16 06:18 ..
-rw-r--r-- 1 root root 1517 Nov 16 06:18 run1_PR-F2-01.txt
-rw-r--r-- 1 root root 6299 Nov 16 06:18 run1_PR-F2-02.txt
===== raw_ai_output/run1_PR-F2-01.txt
```json
[
  {
    "req_id": "REQ-SF-01",
    "rule": "Khách thêm sản phẩm vào giỏ từ trang chi tiết sản phẩm với số lượng chọn được."
  },
  {
    "req_id": "REQ-SF-02",
    "rule": "Sản phẩm hết hàng không được thêm vào giỏ."
  },
  {
    "req_id": "REQ-SF-03",
    "rule": "Số lượng mỗi sản phẩm trong giỏ từ 1 đến 10."
  },
  {
    "req_id": "REQ-SF-04",
    "rule": "Khách thay đổi số lượng hoặc xóa dòng sản phẩm; tổng tiền cập nhật ngay."
  },
  {
    "req_id": "REQ-SF-07",
    "rule": "Giỏ của khách đã đăng nhập được giữ lại khi đăng xuất và đăng nhập lại."
  },
  {
    "req_id": "REQ-SF-10",
    "rule": "Checkout gồm 4 bước: Giỏ hàng → Đăng nhập → Địa chỉ thanh toán → Phương thức thanh toán."
  },
  {
    "req_id": "REQ-SF-11",
    "r

In [5]:
# Don ket qua lan chay loi (max_tokens 2000 bi cat) + kiem tra ban script moi
!rm -rf raw_ai_output f2_llm_calls.csv; grep -n "max_tokens\|reasoning_effort\|FIX =" F2_prompt_chain.py

10:FIX = pathlib.Path("f2_fixtures")                       # Colab: fixture nằm ở /content/f2_fixtures (mẫu gốc: ../f2_fixtures)
14:CFG = {"temperature": 0.2, "max_tokens": 8000,           # giữ cố định giữa các lần chạy để kiểm tra tái lập
16:       # Gemini 3.x là model "thinking": token suy nghĩ tính vào max_tokens -> JSON có thể bị cắt. Tắt thinking, giữ cố định.
17:       "reasoning_effort": "none"}
